# Phụ lục: Bằng chứng và khả năng tái lập

## Nguồn dữ liệu

Báo cáo giữ nguyên artifacts đã nhập; dữ liệu mới được bổ sung thành bộ riêng, không ghi đè ZIP/corpus gốc. SHA-256 được kiểm khi build để bảo đảm bảng dựa đúng tệp đã lưu. Kiểm tra hash artifact không thay cho tái kiểm corpus sống trên Drive. Bộ TV5 hiện dùng ZIP cục bộ cùng tên/kích thước bản Drive, chưa đối chiếu hash cloud.

Nguồn phân công là workbook đăng ký nhóm. Tiến độ “đã chạy 5.000 ID/người” là thông tin nhóm cung cấp khi trao đổi ngày 04/11/2026; đó không phải số được suy từ benchmark hoặc tự tạo từ artifacts.

Thư mục chia sẻ: [arxiv_ms1_data](https://drive.google.com/drive/folders/123TUmjXMX38gKRLnJIKDl2yZF8jK6e9n).

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
sys.path.insert(0, str(ROOT / 'utils'))
from preliminary_results import (load_members, progress_rows, assignment_outcomes,
    quality_rows, error_groups, performance_rows, run_range, session_members, coverage_rows)
members = load_members(ROOT)
sessions = session_members(ROOT, members)
def table(rows):
    frame = pd.DataFrame(rows, dtype=object)
    frame = frame.where(frame.notna(), 'Chưa đủ bằng chứng')
    def number(value):
        if isinstance(value, float):
            return f'{value:,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
        if isinstance(value, int) and not isinstance(value, bool):
            return f'{value:,}'.replace(',', '.')
        return str(value)
    display(HTML(frame.to_html(index=False, escape=True, border=0, classes='dataframe',
                              formatters={column:number for column in frame.columns})))


In [ ]:
for s in sessions:
    p=s['provenance']
    display(Markdown('**'+s['session']+'**: ['+p['archive_name']+']('+p['drive_url']+')'))
    display(Markdown('SHA-256 ZIP: `'+p['archive_sha256']+'`'))
table([{'Phiên': s['session'], 'Run ID': s['benchmark'].get('run_id'),
        'Bắt đầu (UTC)': s['benchmark']['started_at'], 'Kết thúc (UTC)': s['benchmark']['finished_at'],
        'Source commit': s['benchmark']['environment'].get('source_commit'),
        'CPU logic': s['benchmark']['environment'].get('logical_cpu_count'),
        'CPU-only ghi nhận': s['benchmark']['environment'].get('cpu_only_confirmation')} for s in sessions])

## Đối chiếu audit và phạm vi

Trong hai commit source được bộ benchmark chính khai báo (`3985747` và `4ea44c4`), cờ hoàn tất audit yêu cầu không còn ID chưa thấy, Discovery chưa giải quyết, validation fail và diagnostics. Vì vậy cờ này chặt hơn điều kiện “đã chạy qua đủ ID”. Trường `discovered_assigned_ids` ở hai phiên bản đếm ID có record, kể cả lỗi; Discovery thành công được đọc riêng từ bảng trạng thái.

Phạm vi TV2 được xác minh từ đủ 5.000 định danh trong kết quả paper; TV5 được đối chiếu thêm với tệp input đi cùng audit lồng trùng nội dung. TV3 có input dải đầy đủ nhưng ZIP/thư mục audit đã kiểm tra chưa có kết quả audit. Bản TV4 cuối đã lưu và hai ZIP bổ sung chưa chứa audit toàn dải.

TV2 báo `audit_complete=true` đồng thời có 2.579 paper fail, mâu thuẫn công thức source được khai báo. Giữ nguyên giá trị gốc, cần xác minh code thực dùng và chạy lại audit; không chuyển cờ này thành kết luận hoàn tất chất lượng. Snapshot TV5 còn bốn Discovery lỗi: 2410.04689, 2410.04690, 2410.05043, 2410.05044.

In [ ]:
table(coverage_rows(members))

## Vai trò artifacts

| Tệp | Vai trò |
|---|---|
| assignment-audit | Coverage và validation trên dải audit |
| inputs | Cấu hình phạm vi; không chứng minh audit đã chạy |
| statistics | Số liệu theo snapshot/run; phải kiểm tra phạm vi và run ID |
| summary | Tổng hợp pipeline một run; không cộng chồng lặp |
| benchmark, samples, timings | Số đo môi trường, tài nguyên và event theo run |
| validation từng paper | Chi tiết kiểm định cần đối chiếu với dữ liệu thật |
| run | Cấu hình lúc bắt đầu; trạng thái kết thúc đọc trong benchmark |
| provenance | Nguồn ZIP và hash gốc/bản công bố |

Statistics TV2 và hai phiên TV4 bổ sung chưa có snapshot hoàn tất của run hiện tại; lỗi này được giữ trong bộ bằng chứng và không thay bằng manifest cũ. TV3–TV5 ở các run chính có statistics khớp run ID/label; đây là kiểm tra nội bộ artifacts, chưa xác thực lại hash manifest trên Drive.

## Tái lập báo cáo

README của repository báo cáo hướng dẫn cài môi trường và build. Jupyter Book thực thi notebook khi build; các cell tính toán được ẩn khỏi HTML nhưng vẫn lưu trong notebook nguồn. Script `scripts/prepare_submission_report.py` tạo lại nội dung đã biên tập. Hai snapshot artifacts được lưu riêng trong thư mục summaries; không cần gọi arXiv hoặc Drive khi build.

Các tài liệu gốc của thành viên được liên kết ở [Đóng góp thành viên](member-contributions.md). Các mẫu demo/smoke ngày 28/10 được giữ trong repository làm lịch sử và không được cộng vào kết quả thực nghiệm hiện tại.